# Text Preprocessing in NLP — Sarcasm Detection Project

This notebook demonstrates a practical NLP text-cleaning pipeline that can be used before building a sarcasm detection model.


## Objectives

- Understand why text preprocessing is needed.
- Clean noisy social-media-style text.
- Normalize text using lowercasing, URL removal, punctuation handling, and whitespace cleanup.
- Tokenize and remove stopwords.
- Apply stemming and lemmatization.
- Prepare text for a machine-learning sarcasm classifier.


In [ ]:
# Install once if required:
# !pip install pandas numpy nltk scikit-learn

import re
import string
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


## 1. Sample Sarcasm Dataset

For demonstration, we create a small dataset. `1` means sarcastic and `0` means non-sarcastic. In a real project, replace this with your actual dataset.

In [ ]:
data = {
    'text': [
        'Oh great! Another Monday... just what I needed 😂',
        'I absolutely love waiting 3 hours for a bus.',
        'The weather is beautiful today.',
        'What a fantastic update! My app crashes every 2 minutes.',
        'I enjoyed the movie and the story was interesting.',
        'Thanks for the amazing customer service!!! 🙃',
        'Sure, because losing my homework is exactly how I planned my day.',
        'The new library is quiet and comfortable.'
    ],
    'label': [1, 1, 0, 1, 0, 1, 1, 0]
}

df = pd.DataFrame(data)
df

## 2. Inspect the Raw Text

Before cleaning, inspect the text because punctuation, URLs, emojis, repeated characters, hashtags, and slang may carry useful information for sarcasm detection.

In [ ]:
for i, text in enumerate(df['text'], start=1):
    print(f'{i}. {text}')

## 3. Basic Text Cleaning Function

The function below performs common cleaning operations. Notice that emojis are removed here; for a real sarcasm project, you may want to preserve emojis because they can provide sentiment and sarcasm signals.

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)  # URLs
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)              # mentions
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)              # hashtags
    text = re.sub(r'\\d+', '', text)                         # numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\\s+', ' ', text).strip()                # extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']]

## 4. Tokenization

Tokenization converts a sentence into smaller units called tokens, usually words or punctuation marks.

In [ ]:
df['tokens'] = df['clean_text'].apply(word_tokenize)
df[['clean_text', 'tokens']]

## 5. Stopword Removal

Stopwords are common words such as `the`, `is`, `a`, and `to`. Removing them can reduce vocabulary size, but they should not always be removed. In sarcasm detection, small function words and sentence structure can sometimes be useful.

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens_no_stopwords'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stopwords']]

## 6. Stemming

Stemming reduces words to a root-like form. It is fast, but the result may not always be a valid English word.

In [ ]:
stemmer = PorterStemmer()

df['stemmed'] = df['tokens_no_stopwords'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)

df[['tokens_no_stopwords', 'stemmed']]

## 7. Lemmatization

Lemmatization attempts to convert a word to its meaningful dictionary form (lemma). It is usually more linguistically meaningful than stemming.

In [ ]:
lemmatizer = WordNetLemmatizer()

df['lemmatized'] = df['tokens_no_stopwords'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

df[['tokens_no_stopwords', 'lemmatized']]

## 8. Final Preprocessed Text

Join the processed tokens back into strings so they can be passed to vectorizers such as TF-IDF.

In [ ]:
df['processed_text'] = df['lemmatized'].apply(lambda tokens: ' '.join(tokens))
df[['text', 'processed_text', 'label']]

## 9. TF-IDF Feature Extraction

TF-IDF converts text into numerical features. It gives higher importance to words that are frequent in a document but less common across the complete collection.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X = vectorizer.fit_transform(df['processed_text'])
y = df['label']

print('Feature matrix shape:', X.shape)
print('First few features:', vectorizer.get_feature_names_out()[:20])

## 10. Simple Sarcasm Detection Model

This is only a demonstration because the sample dataset is very small. For a real project, use a much larger labeled dataset and a train/test split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# A larger real dataset is recommended. This split is only for demonstration.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))

## 11. Predict New Text

The same preprocessing used during training must be applied to new text.

In [ ]:
def preprocess_for_model(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

new_text = 'Wow, another amazing error message! Just perfect.'
processed = preprocess_for_model(new_text)
new_vector = vectorizer.transform([processed])
prediction = model.predict(new_vector)[0]

print('Original:', new_text)
print('Processed:', processed)
print('Prediction:', 'Sarcastic' if prediction == 1 else 'Not Sarcastic')

## Important Note for a Real Sarcasm Project

Do not blindly remove every punctuation mark, emoji, hashtag, repeated character, or stopword. Sarcasm often depends on context and writing style. Features such as `!!!`, `...`, emojis, capitalization, contrast words, hashtags, and sentence context may contain useful signals.

A production pipeline should also avoid data leakage: fit the vectorizer only on training data, then transform validation/test data with the already-fitted vectorizer.